In [6]:
import pandas as pd
import json
import warnings
warnings.filterwarnings("ignore")

from evidently import Report
from evidently.presets import DataDriftPreset

# 1. Chemins absolus vers tes fichiers
csv_path = r"C:\Users\j-a-b\Documents\OpenClassrooms\Projet 5 PMVL Asset\PMVL\pmvl_cleaned_prepared_with_features.csv"
logs_path = r"C:\Users\j-a-b\Documents\OpenClassrooms\Projet 5 PMVL Asset\logs\prediction_logs.jsonl"

# 2. Charger les données de référence (Entraînement)
print("Chargement des données de référence...")
reference_data = pd.read_csv(csv_path, sep=",", low_memory=False, encoding="latin-1")

# 3. Charger les logs de production
logs = []
print("Chargement des logs de production...")
with open(logs_path, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():  # ignore les lignes vides
            logs.append(json.loads(line))

# 4. Extraire les requêtes réussies et créer le DataFrame de production
successful_logs = [log for log in logs if log.get("status") == "success"]

# Extraire la clé "inputs" (qui contient déjà les colonnes au format "PMVL[...]")
production_inputs = [log.get("inputs", {}) for log in successful_logs]
current_data = pd.DataFrame(production_inputs)

# Normaliser les noms de colonnes (en enlevant les espaces superflus)
reference_data.columns = [c.strip() for c in reference_data.columns]
current_data.columns = [c.strip() for c in current_data.columns]

if current_data.empty:
    print("Attention: Aucune donnée de production valide trouvée dans les logs.")
else:
    # Recalcul des colonnes communes
    common_columns = sorted(set(reference_data.columns).intersection(set(current_data.columns)))

print(f"Colonnes communes trouvées : {len(common_columns)}")
print(common_columns[:20])  # pour vérifier visuellement

if len(common_columns) == 0:
    print("ERREUR : Aucune colonne commune.")
else:
    reference_data_aligned = reference_data[common_columns]
    current_data_aligned = current_data[common_columns]

    # 6. Générer le rapport de Data Drift
    print("Génération du rapport Evidently AI...")
    drift_report = Report(metrics=[DataDriftPreset()])
        
    # On sauvegarde le résultat de la fonction run() dans my_eval
    my_eval = drift_report.run(reference_data=reference_data_aligned, current_data=current_data_aligned)

    # Sauvegarder le rapport sous forme de page HTML interactive via my_eval
    my_eval.save_html("data_drift_report.html")
    print("Rapport sauvegardé : data_drift_report.html")

# 7. Analyser les métriques opérationnelles
latencies = [log.get("latency_ms", 0) for log in successful_logs if "latency_ms" in log]
errors = [log for log in logs if log.get("status") == "error"]

print(f"\n--- Métriques Opérationnelles ---")
print(f"Total des requêtes analysées: {len(logs)}")
if len(logs) > 0:
    print(f"Taux d'erreur: {(len(errors) / len(logs)) * 100:.2f}%")
if latencies:
    print(f"Latence moyenne: {sum(latencies) / len(latencies):.2f} ms")
    print(f"Latence max: {max(latencies):.2f} ms")

Chargement des données de référence...
Chargement des logs de production...
Colonnes communes trouvées : 21
['PMVL[3A]', 'PMVL[CANTON]', 'PMVL[CIC]', 'PMVL[ENTITE]', 'PMVL[GROUPE]', 'PMVL[Holding date]', 'PMVL[ISIN]', 'PMVL[Orig. name]', 'PMVL[PFMIndice.Perf J / J-1]', 'PMVL[PRMP MtM]', 'PMVL[PRMP PMVL]', 'PMVL[PRMP VNC]', 'PMVL[Parametres_Indices.TICKER]', 'PMVL[Ptf name]', 'PMVL[Purch. Val. (clean) (ptf cur.)]', 'PMVL[Quantity]', 'PMVL[Quote Date]', 'PMVL[Quote]', 'PMVL[Ref Unik Asset]', 'PMVL[Selected Fund code]']
Génération du rapport Evidently AI...


Rapport sauvegardé : data_drift_report.html

--- Métriques Opérationnelles ---
Total des requêtes analysées: 3
Taux d'erreur: 0.00%
Latence moyenne: 8.37 ms
Latence max: 10.70 ms


In [3]:
print("Colonnes référence (premières 20) :")
print(list(reference_data.columns)[:20])

print("\nColonnes production (premières 20) :")
print(list(current_data.columns)[:20])

Colonnes référence (premières 20) :
['PMVL[Holding date],PMVL[ENTITE],PMVL[ISIN],PMVL[Orig. name],PMVL[Parametres_Indices.TICKER],PMVL[PFMIndice.Perf J / J-1],PMVL[PRMP PMVL],PMVL[PRMP VNC],PMVL[PRMP MtM],PMVL[Ref Unik Asset],PMVL[Selected Fund code],PMVL[3A],PMVL[CANTON],PMVL[CIC],PMVL[GROUPE],PMVL[Ptf name],PMVL[Quantity],PMVL[Purch. Val. (clean) (ptf cur.)],PMVL[Quote Date],PMVL[Quote],PMVL[VNC Agrege dirty (ptf cur.)],PMVL[PMVL Estimé],PRMP_PMVL_future,target']

Colonnes production (premières 20) :
['PMVL[Holding date]', 'PMVL[Quote Date]', 'PMVL[PMVL Estimé]', 'PMVL[PFMIndice.Perf J / J-1]', 'PMVL[PRMP PMVL]', 'PMVL[PRMP VNC]', 'PMVL[PRMP MtM]', 'PMVL[Quantity]', 'PMVL[Purch. Val. (clean) (ptf cur.)]', 'PMVL[Quote]', 'PMVL[VNC Agrege dirty (ptf cur.)]', 'PMVL[ENTITE]', 'PMVL[ISIN]', 'PMVL[Orig. name]', 'PMVL[Parametres_Indices.TICKER]', 'PMVL[Ref Unik Asset]', 'PMVL[Selected Fund code]', 'PMVL[3A]', 'PMVL[CANTON]', 'PMVL[CIC]']
